# **Problem Statement**

## Business Context

A sales forecast is a prediction of future sales revenue based on historical data, industry trends, and the status of the current sales pipeline. Businesses use the sales forecast to estimate weekly, monthly, quarterly, and annual sales totals. A company needs to make an accurate sales forecast as it adds value across an organization and helps the different verticals to chalk out their future course of action.

Forecasting helps an organization plan its sales operations by region and provides valuable insights to the supply chain team regarding the procurement of goods and materials. An accurate sales forecast process has many benefits which include improved decision-making about the future and reduction of sales pipeline and forecast risks. Moreover, it helps to reduce the time spent in planning territory coverage and establish benchmarks that can be used to assess trends in the future.

## Objective

SuperKart is a retail chain operating supermarkets and food marts across various tier cities, offering a wide range of products. To optimize its inventory management and make informed decisions around regional sales strategies, SuperKart wants to accurately forecast the sales revenue of its outlets for the upcoming quarter.

To operationalize these insights at scale, the company has partnered with a data science firm—not just to build a predictive model based on historical sales data, but to develop and deploy a robust forecasting solution that can be integrated into SuperKart’s decision-making systems and used across its network of stores.

## Data Description

The data contains the different attributes of the various products and stores.The detailed data dictionary is given below.

- **Product_Id** - unique identifier of each product, each identifier having two letters at the beginning followed by a number.
- **Product_Weight** - weight of each product
- **Product_Sugar_Content** - sugar content of each product like low sugar, regular and no sugar
- **Product_Allocated_Area** - ratio of the allocated display area of each product to the total display area of all the products in a store
- **Product_Type** - broad category for each product like meat, snack foods, hard drinks, dairy, canned, soft drinks, health and hygiene, baking goods, bread, breakfast, frozen foods, fruits and vegetables, household, seafood, starchy foods, others
- **Product_MRP** - maximum retail price of each product
- **Store_Id** - unique identifier of each store
- **Store_Establishment_Year** - year in which the store was established
- **Store_Size** - size of the store depending on sq. feet like high, medium and low
- **Store_Location_City_Type** - type of city in which the store is located like Tier 1, Tier 2 and Tier 3. Tier 1 consists of cities where the standard of living is comparatively higher than its Tier 2 and Tier 3 counterparts.
- **Store_Type** - type of store depending on the products that are being sold there like Departmental Store, Supermarket Type 1, Supermarket Type 2 and Food Mart
- **Product_Store_Sales_Total** - total revenue generated by the sale of that particular product in that particular store


# **Installing and Importing the necessary libraries**

In [ ]:
#Installing the libraries with the specified versions
!pip install numpy==2.0.2 pandas==2.2.2 scikit-learn==1.6.1 matplotlib==3.10.0 seaborn==0.13.2 joblib==1.4.2 xgboost==2.1.4 requests==2.32.4 huggingface_hub>=0.34.0 -q

**Note:**

- After running the above cell, kindly restart the notebook kernel (for Jupyter Notebook) or runtime (for Google Colab) and run all cells sequentially from the next cell.

- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in this notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

# Libraries to help with reading and manipulating data
import numpy as np
import pandas as pd

# For splitting the dataset
from sklearn.model_selection import train_test_split

# Libaries to help with data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Removes the limit for the number of displayed columns
pd.set_option("display.max_columns", None)
# Sets the limit for the number of displayed rows
pd.set_option("display.max_rows", 100)


# Libraries different ensemble classifiers
from sklearn.ensemble import (
    BaggingRegressor,
    RandomForestRegressor,
    AdaBoostRegressor,
    GradientBoostingRegressor,
)
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor

# Libraries to get different metric scores
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    mean_absolute_percentage_error
)

# To create the pipeline
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline,Pipeline

# To tune different models and standardize
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler,OneHotEncoder

# To serialize the model
import joblib

# os related functionalities
import os

# API request
import requests

# for hugging face space authentication to upload files
from huggingface_hub import login, HfApi

# **Loading the dataset**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
kart = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/SuperKart.csv")

In [ ]:
#Copy data to another variable to avoid any changes to original data
data = kart.copy()

# **Data Overview**

View first and last 5 rows

In [ ]:
data.head(5)

In [ ]:
data.tail(5)

In [ ]:
print(f"There are {data.shape[0]} rows and {data.shape[1]} columns.")

In [ ]:
#Checking the data types of columns
data.info()

In [ ]:
#checking statitical summary of data
data.describe(include="all").T

In [ ]:
# checking for duplicate values
data.duplicated().sum()

- No duplicate data is seen .

In [ ]:
# checking for missing values in the data
data.isnull().sum()

- No null values are found in the dataset.

# **Exploratory Data Analysis (EDA)**

## Univariate Analysis

In [ ]:
# function to plot a boxplot and a histogram along the same scale.

def histogram_boxplot(data, feature, figsize=(12, 7), kde=False, bins=None):
    """
    Boxplot and histogram combined

    data: dataframe
    feature: dataframe column
    figsize: size of figure (default (12,7))
    kde: whether to the show density curve (default False)
    bins: number of bins for histogram (default None)
    """
    f2, (ax_box2, ax_hist2) = plt.subplots(
        nrows=2,  # Number of rows of the subplot grid= 2
        sharex=True,  # x-axis will be shared among all subplots
        gridspec_kw={"height_ratios": (0.25, 0.75)},
        figsize=figsize,
    )  # creating the 2 subplots
    sns.boxplot(
        data=data, x=feature, ax=ax_box2, showmeans=True, color="violet"
    )  # boxplot will be created and a star will indicate the mean value of the column
    sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2, bins=bins, palette="winter"
    ) if bins else sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2
    )  # For histogram
    ax_hist2.axvline(
        data[feature].mean(), color="green", linestyle="--"
    )  # Add mean to the histogram
    ax_hist2.axvline(
        data[feature].median(), color="black", linestyle="-"
    )  # Add median to the histogram

##Product Weight

In [ ]:
histogram_boxplot(data, "Product_Weight")

- We observe that there is more sales for items with weight ranging between 5.5 to 20 units, with highest between the range if 11 to 13.

##Product_Allocated_Area

In [ ]:
histogram_boxplot(data, "Product_Allocated_Area")

- Products allocated in the initial areas seem to have more sales than the ;ater ones.

##Product_MRP

In [ ]:
histogram_boxplot(data, "Product_MRP")

##Product_Store_Sales_Total

In [ ]:
histogram_boxplot(data, "Product_Store_Sales_Total")

#Bivariate Analysis

In [ ]:
# function to create labeled barplots


def labeled_barplot(data, feature, perc=False, n=None):
    """
    Barplot with percentage at the top

    data: dataframe
    feature: dataframe column
    perc: whether to display percentages instead of count (default is False)
    n: displays the top n category levels (default is None, i.e., display all levels)
    """

    total = len(data[feature])  # length of the column
    count = data[feature].nunique()
    if n is None:
        plt.figure(figsize=(count + 1, 5))
    else:
        plt.figure(figsize=(n + 1, 5))

    plt.xticks(rotation=90, fontsize=15)
    ax = sns.countplot(
        data=data,
        x=feature,
        palette="Paired",
        order=data[feature].value_counts().index[:n].sort_values(),
    )

    for p in ax.patches:
        if perc == True:
            label = "{:.1f}%".format(
                100 * p.get_height() / total
            )  # percentage of each class of the category
        else:
            label = p.get_height()  # count of each level of the category

        x = p.get_x() + p.get_width() / 2  # width of the plot
        y = p.get_height()  # height of the plot

        ax.annotate(
            label,
            (x, y),
            ha="center",
            va="center",
            size=12,
            xytext=(0, 5),
            textcoords="offset points",
        )  # annotate the percentage

    plt.show()

In [ ]:
labeled_barplot(data, "Product_Sugar_Content", perc=True)

- Low Sugar products seem to have high demand while foods with no sugar have less demand. Reg and Regular category here seems to be similar category.

In [ ]:
#Product Type
labeled_barplot(data, "Product_Type", perc=True)

- Fruits and Vegetables are the product type with high demand/sales with Snack foods as the next category.
- Seafood and breakfast items have lowest demand.

In [ ]:
labeled_barplot(data, "Store_Id", perc=True)

- OUT004 CATEGORY has highest sales whereas OUT002 has lowest sales.

In [ ]:
labeled_barplot(data, "Store_Size", perc=True)

- Stores with mediun size seems to have highest store sales whereas small size stores have lowest store sales.

In [ ]:
labeled_barplot(data, "Store_Location_City_Type", perc=True)

- Tier 2 city type seem to have highest sales compared to Tier1 . Tier3 has lowest sales.

In [ ]:
labeled_barplot(data, "Store_Type", perc=True)

- Supermarket Type 2 seem to have high store sales, with Supermarket Type1 being the second.
- Foodmart is the categoty with lowest store sales.

#Correlation Matrix


In [ ]:
cols_list = data.select_dtypes(include=np.number).columns.tolist()

plt.figure(figsize=(10, 5))
sns.heatmap(
    data[cols_list].corr(), annot=True, vmin=-1, vmax=1, fmt=".2f", cmap="Spectral"
)
plt.show()

- Higher-priced products(Product_MRP) tend to generate higher total sales value.
- Heavier products(Product_weight) are associated with higher sales.

Let's check the distribution of our target variable i.e Product_Store_Sales_Total with the numeric columns

In [ ]:
plt.figure(figsize=[8, 6])
sns.scatterplot(x=data.Product_Weight, y=data.Product_Store_Sales_Total)
plt.show()

In [ ]:
plt.figure(figsize=[8, 6])
sns.scatterplot(x="Product_Allocated_Area", y="Product_Store_Sales_Total", data=data)
plt.show()

In [ ]:
plt.figure(figsize=[8, 6])
sns.scatterplot(x="Product_MRP", y="Product_Store_Sales_Total", data=data)
plt.show()

Let us see from which product type the company is generating most of the revenue

In [ ]:
df_revenue1 = data.groupby(["Product_Type"], as_index=False)[
    "Product_Store_Sales_Total"
].sum()
plt.figure(figsize=[14, 8])
plt.xticks(rotation=90)
a = sns.barplot(x=df_revenue1.Product_Type, y=df_revenue1.Product_Store_Sales_Total)
a.set_xlabel("Product Types")
a.set_ylabel("Revenue")
plt.show()

In [ ]:
#the code to perform a groupby on Product_Sugar_Content and select Product_Store_Sales_Total
df_revenue2 = data.groupby(["Product_Sugar_Content"], as_index=False)[
    "Product_Store_Sales_Total"
].sum()
plt.figure(figsize=[8, 6])
plt.xticks(rotation=90)
b = sns.barplot(
    x=df_revenue2.Product_Sugar_Content, y=df_revenue2.Product_Store_Sales_Total
)
b.set_xlabel("Product_Sugar_content")
b.set_ylabel("Revenue")
plt.show()

- Low sugar content products seem to have higher sales.
- No sugar and reg products seem have lower sales.

###**Let us see from which type of stores and locations the revenue generation is more.**

In [ ]:
df_store_revenue = data.groupby(["Store_Id"], as_index=False)[
    "Product_Store_Sales_Total"
].sum()
plt.figure(figsize=[8, 6])
plt.xticks(rotation=90)
r = sns.barplot(
    x=df_store_revenue.Store_Id, y=df_store_revenue.Product_Store_Sales_Total
)
r.set_xlabel("Stores")
r.set_ylabel("Revenue")
plt.show()

In [ ]:
#Complete the code to perform a groupby on Store_Size and select Product_Store_Sales_Total
df_revenue3 = data.groupby(["Store_Size"], as_index=False)[
    "Product_Store_Sales_Total"
].sum()
plt.figure(figsize=[8, 6])
plt.xticks(rotation=90)
c = sns.barplot(x=df_revenue3.Store_Size, y=df_revenue3.Product_Store_Sales_Total)
c.set_xlabel("Store_Size")
c.set_ylabel("Revenue")
plt.show()

- Medium store affects the store sales and has highest store sales compared to bigger and small store seizes.

In [ ]:
#code to perform a groupby on Store_Location_City_Type and select Product_Store_Sales_Total
df_revenue4 = data.groupby(["Store_Location_City_Type"], as_index=False)[
    "Product_Store_Sales_Total"
].sum()
plt.figure(figsize=[8, 6])
plt.xticks(rotation=90)
d = sns.barplot(
    x=df_revenue4.Store_Location_City_Type, y=df_revenue4.Product_Store_Sales_Total
)
d.set_xlabel("Store_Location_City_Type")
d.set_ylabel("Revenue")
plt.show()

- Tier 2 city type location has highest store sales.
- Tier 3 city type has lowest store sales, probably based on the population in the city.

In [ ]:
#Code to perform a groupby on Store_Type and select Product_Store_Sales_Total
df_revenue5 = data.groupby(["Store_Type"], as_index=False)[
    "Product_Store_Sales_Total"
].sum()
plt.figure(figsize=[8, 6])
plt.xticks(rotation=90)
e = sns.barplot(x=df_revenue5.Store_Type, y=df_revenue5.Product_Store_Sales_Total)
e.set_xlabel("Store_Type")
e.set_ylabel("Revenue")
plt.show()

- Supermarket Type2 has highest store sales , followed bt Department store.
- Foodmart has the lowest store sales .

### **Let's check the distribution of our target variable i.e Product_Store_Sales_Total with the other categorical columns**

In [ ]:
plt.figure(figsize=[14, 8])
sns.boxplot(data=data, x="Store_Id", y="Product_Store_Sales_Total", hue = "Store_Id")
plt.xticks(rotation=90)
plt.title("Boxplot - Store_Id Vs Product_Store_Sales_Total")
plt.xlabel("Stores")
plt.ylabel("Product_Store_Sales_Total (of each product)")
plt.show()

- OUT003 and OUT001 seem to have higher sales with no outliers.
- OU004 and OUT002 have outliers .

In [ ]:
plt.figure(figsize=[14, 8])
sns.boxplot(data = data, x = "Store_Size", y = "Product_Store_Sales_Total", hue = "Store_Size")
plt.xticks(rotation=90)
plt.title("Boxplot - Store_Size Vs Product_Store_Sales_Total")
plt.xlabel("Stores")
plt.ylabel("Product_Store_Sales_Total (of each product)")
plt.show()

- High store size seem to have sales in the range of 3000-4500 whereas small store size seem to have less sales.
- Small store size seem to have some outlier data in lower range.
- Medium store size has sales between 300-4000 with outliers in the higher range .

### **Let's now try to find out some relationship between the other columns**

In [ ]:
plt.figure(figsize=[14, 8])
sns.boxplot(data = data, x = "Product_Type", y = "Product_Weight", hue = "Product_Type")
plt.xticks(rotation=90)
plt.title("Boxplot - Product_Type Vs Product_Weight")
plt.xlabel("Types of Products")
plt.ylabel("Product_Weight")
plt.show()

### **Let's find out whether there is some relationship between the weight of the product and its sugar content**

In [ ]:
plt.figure(figsize=[14, 8])
sns.boxplot(data = data, x = "Product_Sugar_Content", y = "Product_Weight", hue = "Product_Sugar_Content")
plt.xticks(rotation=90)
plt.title("Boxplot - Product_Sugar_Content Vs Product_Weight")
plt.xlabel("Product_Sugar_Content")
plt.ylabel("Product_Weight")
plt.show()

- There is no major difference in product weight across sugar content categories. Any differences are small and mainly visible in the number and direction of outliers rather than in the central tendency.

### **Let's analyze the sugar content of different product types**

In [ ]:
plt.figure(figsize=(14, 8))
sns.heatmap(
    pd.crosstab(data["Product_Sugar_Content"], data["Product_Type"]),
    annot=True,
    fmt="g",
    cmap="viridis",
)
plt.ylabel("Product_Sugar_Content")
plt.xlabel("Product_Type")
plt.show()

- Heatmap shows strong skew towards Low sugar products, they dominate many categories like fruits & vegetables, snack foods, Dairy etc.
- No sugar are mostly non-food items emphasizing on Lower sugar options in terms of demand.

###**To check how many items of each product type has been sold in each of the stores**

In [ ]:
plt.figure(figsize=(14, 8))
sns.heatmap(
    pd.crosstab(data["Store_Id"], data["Product_Type"]),
    annot=True,
    fmt="g",
    cmap="viridis",
)
plt.ylabel("Stores")
plt.xlabel("Product_Type")
plt.show()

- OUT004 is a major outlet with broad and deep inventory.
- OUT001-OUT003 are smaller stores with limited but balanced product ranges.
- The heatmap highlights both store size differences and consumer demand patterns across product types.

### **Different product types have different prices. Let's analyze the trend.**

In [ ]:
plt.figure(figsize=[14, 8])
sns.boxplot(data = data, x = "Product_Type", y = "Product_MRP", hue = "Product_Type")
plt.xticks(rotation=90)
plt.title("Boxplot - Product_Type Vs Product_MRP")
plt.xlabel("Product_Type")
plt.ylabel("Product_MRP (of each product)")
plt.show()

### **Let's find out how the Product_MRP varies with the different stores**

In [ ]:
plt.figure(figsize=[14, 8])
sns.boxplot(data = data, x = "Store_Id", y = "Product_MRP", hue = "Store_Id")
plt.xticks(rotation=90)
plt.title("Boxplot - Store_Id Vs Product_MRP")
plt.xlabel("Stores")
plt.ylabel("Product_MRP (of each product)")
plt.show()

### **Let's look deeper and do a detailed analysis of each of the stores**.

#### OUT001

In [ ]:
data.loc[data["Store_Id"] == "OUT001"].describe(include="all").T

- OUT001 is a store of Supermarket Type 1 which is located in a Tier 2 city and has store size as high. It was established in 1987.
- OUT001 has sold products whose MRP range from 71 to 227.
- Snack Foods have been sold the highest number of times in OUT001.
- The revenue generated from each product at OUT001 ranges from 2300 to 5000.

In [ ]:
data.loc[data["Store_Id"] == "OUT001", "Product_Store_Sales_Total"].sum()

*  OUT001 has generated total revenue of 6223113 from the sales of goods.


In [ ]:
df_OUT001 = (
    data.loc[data["Store_Id"] == "OUT001"]
    .groupby(["Product_Type"], as_index=False)["Product_Store_Sales_Total"]
    .sum()
)
plt.figure(figsize=[14, 8])
plt.xticks(rotation=90)
plt.xlabel("Product_Type")
plt.ylabel("Product_Store_Sales_Total")
plt.title("OUT001")
sns.barplot(x=df_OUT001.Product_Type, y=df_OUT001.Product_Store_Sales_Total)
plt.show()

- OUT001 has generated the highest revenue from the sale of fruits and vegetables and snack foods. Both the categories have contributed around 800000 each.

#### OUT002

In [ ]:
data.loc[data["Store_Id"] == "OUT002"].describe(include="all").T

- OUT002 is a food mart which is located in a Tier 3 city and has store size as small. It was established in 1998.
- OUT002 has sold products whose MRP range from 31 to 225.
- Fruits and vegetables have been sold the highest number of times in OUT002.
- The revenue generated from each product at OUT002 ranges from 33 to 2300.

In [ ]:
data.loc[data["Store_Id"] == "OUT002", "Product_Store_Sales_Total"].sum()

OUT002 has generated total revenue of 2030910 from the sales of goods.

In [ ]:
df_OUT002 = (
    data.loc[data["Store_Id"] == "OUT002"]
    .groupby(["Product_Type"], as_index=False)["Product_Store_Sales_Total"]
    .sum()
)
plt.figure(figsize=[14, 8])
plt.xticks(rotation=90)
plt.xlabel("Product_Type")
plt.ylabel("Product_Store_Sales_Total")
plt.title("OUT002")
sns.barplot(x=df_OUT002.Product_Type, y=df_OUT002.Product_Store_Sales_Total)
plt.show()

- OUT002 has generated the highest revenue from the sale of fruits and vegetables (~ 300000) followed by snack foods (~ 250000).

#### OUT003

In [ ]:
data.loc[data["Store_Id"] == "OUT003"].describe(include="all").T

**Observations**
- OUT003 is a Departmental store which is located in a Tier 1 city and has store size as medium. It was established in 1999.
- OUT003 has sold products whose MRP range from 86 to 266.
- Snack Foods have been sold the highest number of times in OUT003.
- The revenue generated from each product at OUT003 ranges from 3070 to 8000.

In [ ]:
data.loc[data["Store_Id"] == "OUT003", "Product_Store_Sales_Total"].sum()

OUT003 has generated total revenue of 6673458 from the sales of goods

In [ ]:
df_OUT003 = (
    data.loc[data["Store_Id"] == "OUT003"]
    .groupby(["Product_Type"], as_index=False)["Product_Store_Sales_Total"]
    .sum()
)
plt.figure(figsize=[14, 8])
plt.xticks(rotation=90)
plt.xlabel("Product_Type")
plt.ylabel("Product_Store_Sales_Total")
plt.title("OUT003")
sns.barplot(x=df_OUT003.Product_Type, y=df_OUT003.Product_Store_Sales_Total)
plt.show()

- OUT003 has generated the highest revenue from the sale of snack foods followed by fruits and vegetables, both the categories contributing around 800000 each.

#### OUT004

In [ ]:
data.loc[data["Store_Id"] == "OUT004"].describe(include="all").T

- OUT004 is a store of Supermarket Type2 which is located in a Tier 2 city and has store size as medium. It was established in 2009.
- OUT004 has sold products whose MRP range from 83 to 198.
- Fruits and vegetables have been sold the highest number of times in OUT004.
- The revenue generated from each product at OUT004 ranges from 1561 to 5463.

In [ ]:
data.loc[data["Store_Id"] == "OUT004", "Product_Store_Sales_Total"].sum()

OUT004 has generated total revenue of 15427583 from the sales of goods which is highest among all the 4 stores.

In [ ]:
df_OUT004 = (
    data.loc[data["Store_Id"] == "OUT004"]
    .groupby(["Product_Type"], as_index=False)["Product_Store_Sales_Total"]
    .sum()
)
plt.figure(figsize=[14, 8])
plt.xticks(rotation=90)
plt.xlabel("Product_Type")
plt.ylabel("Product_Store_Sales_Total")
plt.title("OUT004")
sns.barplot(x=df_OUT004.Product_Type, y=df_OUT004.Product_Store_Sales_Total)
plt.show()

- OUT004 has generated the highest revenue from the sale of fruits and vegetables (~ 2500000) followed by snack foods (~ 2000000).

**Let's find out the revenue generated by the stores from each of the product types**.

In [ ]:
df1 = data.groupby(["Product_Type", "Store_Id"], as_index=False)[
    "Product_Store_Sales_Total"
].sum()
df1

- In all the product types, the revenue generated by OUT004 has been the highest which seems quite logical since around 53% of the total products were brought from this store.
- In all the product categories, the revenue generated by OUT002 has been the lowest which seems quite obvious since it is small store in a Tier 3 city.

**Let's find out the revenue generated by the stores from products having different levels of sugar content**.

In [ ]:
df2 = data.groupby(["Product_Sugar_Content", "Store_Id"], as_index=False)[
    "Product_Store_Sales_Total"
].sum()
df2

- The trend is the same as that which was present in the revenue analysis of stores for product types.

# **Data Preprocessing**

## **Replacing the values in the Product_Sugar_Content column**

We observe that in the Product_Sugar_Content column, there are 3 types - Low Sugar, Regular and reg.
It looks like Regular and reg are referring to the same category. So let's replace reg with Regular.

In [ ]:
# Replacing reg with Regular
data.Product_Sugar_Content.replace(to_replace=["reg"], value=["Regular"], inplace=True)

In [ ]:
data.Product_Sugar_Content.value_counts()

## **Exploring Patterns in Product_IDs**

We can see that the Product_Id column has two characters followed by a number.
Let's delve deeper and see whether they are having any relationship with the other columns or not

In [ ]:
## extracting the first two characters from the Product_Id column and storing it in another column
data["Product_Id_char"] = data["Product_Id"].str[:2]
data.head()

In [ ]:
data["Product_Id_char"].unique()

In [ ]:
data.loc[data.Product_Id_char == "FD", "Product_Type"].unique()

In [ ]:
data.loc[data.Product_Id_char == "NC", "Product_Type"].unique()

In [ ]:
data.loc[data.Product_Id_char == "DR", "Product_Type"].unique()

## **Store's Age**

A store which has been in the business for a long duration is more trustworthy than the newly established ones.
On the other hand, older stores may sometimes lack infrastructure if proper attention is not given. So let us calculate the current age of the store and incorporate that in our model.

In [ ]:
data["Store_Age_Years"] = 2025 - data.Store_Establishment_Year

Grouping Product Types into Perishables and Non-Perishables.

We have 16 different product types in our dataset.

So let us make two broad categories, perishables and non perishables, in order to reduce the number of product types

In [ ]:
perishables = [
    "Dairy",
    "Meat",
    "Fruits and Vegetables",
    "Breakfast",
    "Breads",
    "Seafood",
]

In [ ]:
def change(x):
    if x in perishables:
        return "Perishables"
    else:
        return "Non Perishables"

In [ ]:
data['Product_Type_Category'] = data['Product_Type'].apply(change)

In [ ]:
data.head()

#Outlier Check

In [ ]:
# outlier detection using boxplot
numeric_columns = data.select_dtypes(include=np.number).columns.tolist()
numeric_columns.remove("Store_Establishment_Year")
numeric_columns.remove("Store_Age_Years")


plt.figure(figsize=(15, 12))

for i, variable in enumerate(numeric_columns):
    plt.subplot(4, 4, i + 1)
    plt.boxplot(data[variable], whis=1.5)
    plt.tight_layout()
    plt.title(variable)

plt.show()

- We see that Product_weight , Product_Allocated_Area , Product_MRP , Product_Store_Sales_Total have outliers.

## **Data Preparation for Modeling**

- Our aim to forecast the Product_Store_Sales_Total.
- Before building the model, we will drop unnecessary columns and encode the categorical features.
- We will then split the data into training and testing sets to evaluate the model's performance on unseen data.

In [ ]:
data.head()

In [ ]:
#Removing columns that are not required.
data = data.drop(["Product_Id","Product_Type","Store_Id","Store_Establishment_Year"], axis=1) #Complete the code to drop the columns "Product_Id","Product_Type","Store_Id","Store_Establishment_Year"

In [ ]:
data.shape

In [ ]:
data.head()

In [ ]:
# Separating features and the target column
X = data.drop("Product_Store_Sales_Total", axis=1)
y = data["Product_Store_Sales_Total"]

In [ ]:
# Splitting the data into train and test sets in 70:30 ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=1, shuffle=True
)

In [ ]:
X_train.shape, X_test.shape

#Data Preprocessing

In [ ]:
categorical_features = data.select_dtypes(include=['object', 'category']).columns.tolist()
categorical_features

In [ ]:
# Create a preprocessing pipeline for the categorical features
preprocessor = make_column_transformer(
    (Pipeline([('encoder', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
)

# **Model Building**

- Model Building can be done using any one model but I have written code for all six models ,for my understanding.
- I have fit different models on the train data and observe their performance.
- Further written code to improve that performance by tuning some hyperparameters available for that algorithm.
- I have used GridSearchCv for hyperparameter tuning and r_2 score to optimize the model.
- R-square - Coefficient of determination is used to evaluate the performance of a regression model. It is the amount of the variation in the output dependent attribute which is predictable from the input independent variables.
- Starting by creating a function to get model scores, to avoid code redundancy.

### Define functions for Model Evaluation

In [ ]:
# function to compute adjusted R-squared
def adj_r2_score(predictors, targets, predictions):
    r2 = r2_score(targets, predictions)
    n = predictors.shape[0]
    k = predictors.shape[1]
    return 1 - ((1 - r2) * (n - 1) / (n - k - 1))


# function to compute different metrics to check performance of a regression model
def model_performance_regression(model, predictors, target):
    """
    Function to compute different metrics to check regression model performance

    model: regressor
    predictors: independent variables
    target: dependent variable
    """

    # predicting using the independent variables
    pred = model.predict(predictors)

    r2 = r2_score(target, pred)  # to compute R-squared
    adjr2 = adj_r2_score(predictors, target, pred)  # to compute adjusted R-squared
    rmse = np.sqrt(mean_squared_error(target, pred))  # to compute RMSE
    mae = mean_absolute_error(target, pred)  # to compute MAE
    mape = mean_absolute_percentage_error(target, pred)  # to compute MAPE

    # creating a dataframe of metrics
    df_perf = pd.DataFrame(
        {
            "RMSE": rmse,
            "MAE": mae,
            "R-squared": r2,
            "Adj. R-squared": adjr2,
            "MAPE": mape,
        },
        index=[0],
    )

    return df_perf

The ML models to be built can be any two out of the following:
1. Decision Tree
2. Bagging
3. Random Forest
4. AdaBoost
5. Gradient Boosting
6. XGBoost

##Decision Tree

In [ ]:
dtree = DecisionTreeRegressor(random_state=1)
dtree = make_pipeline(preprocessor,dtree)
dtree.fit(X_train, y_train)

Checking model performance on Training Set

In [ ]:
dtree_model_train_perf = model_performance_regression(dtree, X_train, y_train)
dtree_model_train_perf

- Model correctly predicts sales trends about 68.5% of the time
- Typical prediction error is 16.6% or 469-597
- Suitable for planning and forecasting, but expect some variance.

Checking model performance in Test Set

In [ ]:
dtree_model_test_perf = model_performance_regression(dtree, X_test, y_test)
dtree_model_test_perf

Overall,the model explains a good portion of variance with moderate prediction error.

##Bagging Regressor

In [ ]:
bagging_regressor = BaggingRegressor(random_state=1)
bagging_regressor = make_pipeline(preprocessor,bagging_regressor)
bagging_regressor.fit(X_train, y_train)

Checking model performance on Training Set

In [ ]:
bagging_regressor_model_train_perf = model_performance_regression(bagging_regressor, X_train, y_train)
bagging_regressor_model_train_perf

- This model performs better with lower errors and higher explanatory power.
- Based on R-squared value, the model explains 68.5% of the variance, indicating a better fit.

Checking model performance on Test Set

In [ ]:
bagging_regressor_model_test_perf = model_performance_regression(bagging_regressor, X_test, y_test)
bagging_regressor_model_test_perf

- The model shows reasonable explanatory power but higher prediction errors compared to better-performing alternatives.

##Random Forest

In [ ]:
rf_estimator = RandomForestRegressor(random_state=1)
rf_estimator = make_pipeline(preprocessor,rf_estimator)
rf_estimator.fit(X_train, y_train)

Checking model performance on training set

In [ ]:
rf_estimator_model_train_perf = model_performance_regression(rf_estimator, X_train, y_train)
rf_estimator_model_train_perf

Checking model performance on test set

In [ ]:
rf_estimator_model_test_perf = model_performance_regression(rf_estimator, X_test, y_test)
rf_estimator_model_test_perf

##AdaBoost

In [ ]:
ab_regressor = AdaBoostRegressor(random_state=1)
ab_regressor = make_pipeline(preprocessor,ab_regressor)
ab_regressor.fit(X_train, y_train)

Checking model performance on training set

In [ ]:
ab_regressor_model_train_perf = model_performance_regression(ab_regressor, X_train, y_train)
ab_regressor_model_train_perf

Checking model performance on test set

In [ ]:
ab_regressor_model_test_perf = model_performance_regression(ab_regressor, X_test, y_test)
ab_regressor_model_test_perf

##Gradient Boosting Regressor

In [ ]:
gb_estimator = GradientBoostingRegressor(random_state=1)
gb_estimator = make_pipeline(preprocessor,gb_estimator)
gb_estimator.fit(X_train, y_train)

Checking model performance on training set

In [ ]:
gb_estimator_model_train_perf = model_performance_regression(gb_estimator, X_train, y_train)
gb_estimator_model_train_perf

Checking model performance on test set

In [ ]:
gb_estimator_model_test_perf = model_performance_regression(gb_estimator, X_test, y_test)
gb_estimator_model_test_perf

##XGBoost

In [ ]:
xgb_estimator = XGBRegressor(random_state=1)
xgb_estimator = make_pipeline(preprocessor,xgb_estimator)
xgb_estimator.fit(X_train, y_train)

Checking model performance on training set

In [ ]:
xgb_estimator_model_train_perf = model_performance_regression(xgb_estimator, X_train, y_train)
xgb_estimator_model_train_perf

- Overall , This model performs well, with lower errors and stronger explanatory power compared to earlier results.

Checking model performance on test set

In [ ]:
xgb_estimator_model_test_perf = model_performance_regression(xgb_estimator, X_test, y_test)
xgb_estimator_model_test_perf

- The model shows reasonable fit but higher error compared to better-performing versions.

# **Model Performance Improvement - Hyperparameter Tuning**

## Hyperparameter Tuning - Decision Tree

In [ ]:
dtree_tuned = DecisionTreeRegressor(random_state=1)
dtree_tuned = make_pipeline(preprocessor,dtree_tuned)

#Grid of parameters to choose from
parameters = {
    "decisiontreeregressor__max_depth": list(np.arange(2, 6)),
    "decisiontreeregressor__min_samples_leaf": [1, 3, 5],
    "decisiontreeregressor__max_leaf_nodes": [2, 3, 5, 10, 15],
    "decisiontreeregressor__min_impurity_decrease": [0.001, 0.01, 0.1],
}

#Run the grid search
grid_obj = GridSearchCV(dtree_tuned, parameters, scoring=r2_score, cv=3, n_jobs =-1)
grid_obj = grid_obj.fit(X_train, y_train)

#Set the clf to the best combination of parameters
dtree_tuned = grid_obj.best_estimator_

#Fit the best algorithm to the data.
dtree_tuned.fit(X_train, y_train)

### Checking model performance on training set

In [ ]:
dtree_tuned_model_train_perf = model_performance_regression(dtree_tuned, X_train, y_train)
dtree_tuned_model_train_perf

### Checking model performance on test set

In [ ]:
dtree_tuned_model_test_perf = model_performance_regression(dtree_tuned, X_test, y_test)
dtree_tuned_model_test_perf

## Hyperparameter Tuning - Random Forest

In [ ]:
rf_tuned = RandomForestRegressor(random_state=1)
rf_tuned = make_pipeline(preprocessor,rf_tuned)

#Grid of parameters to choose from
parameters = {
    "randomforestregressor__max_depth": [10, 20, None],
    "randomforestregressor__max_features": ['sqrt', 'log2'],
    "randomforestregressor__n_estimators": [100, 200]
}

#Run the grid search
grid_obj = GridSearchCV(rf_tuned, parameters, scoring=r2_score, cv=3, n_jobs = -1)
grid_obj = grid_obj.fit(X_train, y_train)

#Set the clf to the best combination of parameters
rf_tuned = grid_obj.best_estimator_

#Fit the best algorithm to the data.
rf_tuned.fit(X_train, y_train)

### Checking model performance on training set

In [ ]:
rf_tuned_model_train_perf = model_performance_regression(rf_tuned, X_train, y_train)
rf_tuned_model_train_perf

### Checking model performance on training set

In [ ]:
rf_tuned_model_test_perf = model_performance_regression(rf_tuned, X_test, y_test)
rf_tuned_model_test_perf

## Hyperparameter Tuning - Bagging

In [ ]:
# Choose the type of regressor.
bagging_estimator_tuned = BaggingRegressor(random_state=1)
bagging_estimator_tuned = make_pipeline(preprocessor,bagging_estimator_tuned)

# Grid of parameters to choose from
parameters = {
    "baggingregressor__n_estimators":[10,50,100],
    "baggingregressor__max_samples":[0.5,0.7,1.0],
    "baggingregressor__max_features": [0.5, 0.7, 1.0]
}

# Run the grid search
grid_obj = GridSearchCV(bagging_estimator_tuned, parameters, scoring=r2_score, cv=3, n_jobs = -1)
grid_obj = grid_obj.fit(X_train, y_train)

# Set the clf to the best combination of parameters
bagging_estimator_tuned = grid_obj.best_estimator_

# Fit the best algorithm to the data.
bagging_estimator_tuned.fit(X_train, y_train)

### Checking model performance on training set

In [ ]:
bagging_estimator_tuned_model_train_perf = model_performance_regression(bagging_estimator_tuned, X_train, y_train)
bagging_estimator_tuned_model_train_perf

### Checking model performance on testing set

In [ ]:
bagging_estimator_tuned_model_test_perf = model_performance_regression(bagging_estimator_tuned, X_test, y_test)
bagging_estimator_tuned_model_test_perf

## Hyperparameter Tuning - AdaBoost Regressor

In [ ]:
ab_tuned = AdaBoostRegressor(random_state=1)
ab_tuned = make_pipeline(preprocessor,ab_tuned)

# Grid of parameters
parameters = {
    "adaboostregressor__n_estimators": [50,100,200],
    "adaboostregressor__learning_rate": [0.01, 0.1, 0.5, 1.0],
}

# Run the grid search
grid_obj = GridSearchCV(ab_tuned, parameters, scoring=r2_score, cv=3, n_jobs = -1)
grid_obj = grid_obj.fit(X_train, y_train)

# Set the clf to the best combination of parameters
ab_tuned = grid_obj.best_estimator_

# Fit the best algorithm to the data.
ab_tuned.fit(X_train, y_train)

### Checking model performance on training set

In [ ]:
ab_tuned_model_train_perf = model_performance_regression(ab_tuned, X_train, y_train)
ab_tuned_model_train_perf

###Checking model performance on test set

In [ ]:
ab_tuned_model_test_perf = model_performance_regression(ab_tuned, X_test, y_test)
ab_tuned_model_train_perf

## Hyperparameter Tuning - Gradient Boosting

In [ ]:
gb_tuned = GradientBoostingRegressor(random_state=1)
gb_tuned = make_pipeline(preprocessor,gb_tuned)

# Grid of parameters
parameters = {
    "gradientboostingregressor__n_estimators": [50,100,200],
    "gradientboostingregressor__learning_rate": [0.01, 0.1, 0.2],
    "gradientboostingregressor__max_features": ["log2","sqrt"],
    "gradientboostingregressor__max_depth": [3,4,5]
}


# Run the grid search
grid_obj = GridSearchCV(gb_tuned, parameters, scoring=r2_score, cv=3, n_jobs = -1)
grid_obj = grid_obj.fit(X_train, y_train)

# Set the clf to the best combination of parameters
gb_tuned = grid_obj.best_estimator_

# Fit the best algorithm to the data.
gb_tuned.fit(X_train, y_train)

### Checking model performance on training set

In [ ]:
gb_tuned_model_train_perf = model_performance_regression(gb_tuned, X_train, y_train)
gb_tuned_model_train_perf

### Checking model performance on test set

In [ ]:
gb_tuned_model_test_perf = model_performance_regression(gb_tuned, X_test, y_test)
gb_tuned_model_test_perf

## Hyperparameter Tuning - XGBoost Regressor

In [ ]:
# Choose the type of classifier.
xgb_tuned = XGBRegressor(random_state=1)
xgb_tuned = make_pipeline(preprocessor,xgb_tuned)

# Grid of parameters to choose from
parameters = {
    "xgbregressor__n_estimators": [100, 200, 300],
    "xgbregressor__subsample": [0.7, 0.8, 0.9],
    "xgbregressor__gamma": [0, 0.1, 0.2],
    "xgbregressor__colsample_bytree": [0.7, 0.8, 0.9],
    "xgbregressor__colsample_bylevel": [0.7, 0.8, 0.9],
}

# Run the grid search
grid_obj = GridSearchCV(xgb_tuned, parameters, scoring=r2_score, cv=3, n_jobs = -1)
grid_obj = grid_obj.fit(X_train, y_train)

# Set the clf to the best combination of parameters
xgb_tuned = grid_obj.best_estimator_

# Fit the best algorithm to the data.
xgb_tuned.fit(X_train, y_train)

### Checking model performance on training set

In [ ]:
xgb_tuned_model_train_perf = model_performance_regression(xgb_tuned, X_train, y_train)
xgb_tuned_model_train_perf

### Checking model performance on test set

In [ ]:
xgb_tuned_model_test_perf = model_performance_regression(xgb_tuned, X_test, y_test)
xgb_tuned_model_test_perf

# **Model Performance Comparison, Final Model Selection, and Serialization**

In [ ]:
 # training performance comparison
models_train_comp_df = pd.concat(
    [
        xgb_estimator_model_train_perf.T,
        xgb_tuned_model_test_perf.T,
        gb_estimator_model_train_perf.T,
        gb_tuned_model_train_perf.T,
    ],
    axis=1,
)

models_train_comp_df.columns = ["XGB","XGB_Tuned","Gradient","Gradient_Tuned"]

print("Training performance comparison:")
models_train_comp_df

- XGB is the best overall performer with highest R-squared, low RMSE,MAE.It provides better balance of accuracy and explanantory power.
- Gradient Boost has identical performance as XGB.
- Tuning does not guarantee better performance—both tuned models perform worse.
- XGB and Gradient (untuned) are the most reliable choices.


In [ ]:
# Create a folder for storing the files needed for web app deployment
os.makedirs("backend_files", exist_ok=True)

In [ ]:
# Define the file path to save (serialize) the trained model along with the data preprocessing steps
saved_model_path = "backend_files/xgb_tuned_model.joblib"

In [ ]:
# Save the best trained model pipeline using joblib
joblib.dump(xgb_tuned, saved_model_path)
print(f"Model saved successfully at {saved_model_path}")

In [ ]:
# Load the saved model pipeline from the file
saved_model = joblib.load("backend_files/xgb_tuned_model.joblib")

# Confirm the model is loaded
print("Model loaded successfully.")

In [ ]:
saved_model

In [ ]:
#To confirm if the model can now be directly used to make predictions
saved_model.predict(X_test)

# **Deployment - Backend**

## Flask Web Framework


In [ ]:
%%writefile backend_files/app.py

# Import necessary libraries
import numpy as np
import joblib  # For loading the serialized model
import pandas as pd  # For data manipulation
from flask import Flask, request, jsonify  # For creating the Flask API

# Initialize Flask app with a name
superkart_api = Flask("SuperKart Model")

# Load the trained churn prediction model
model = joblib.load("xgb_tuned_model.joblib")

# Define a route for the home page
@superkart_api.get('/')
def home():
    return "Welcome to SuperKart page"
# Define an endpoint to predict churn for a single customer
@superkart_api.post('/v1/predict')
def predict_sales():
    # Get JSON data from the request
    data = request.get_json(force=True)

    # Extract relevant customer features from the input data. The order of the column names matters.
    sample = {
        'Product_Weight': data['Product_Weight'],
        'Product_Sugar_Content': data['Product_Sugar_Content'],
        'Product_Allocated_Area': data['Product_Allocated_Area'],
        'Product_MRP': data['Product_MRP'],
        'Store_Size': data['Store_Size'],
        'Store_Location_City_Type': data['Store_Location_City_Type'],
        'Store_Type': data['Store_Type'],
        'Product_Id_char': data['Product_Id_char'],
        'Store_Age_Years': data['Store_Age_Years'],
        'Product_Type_Category': data['Product_Type_Category']
    }

    # Convert the extracted data into a DataFrame
    input_data = pd.DataFrame([sample])

    # Make a churn prediction using the trained model
    prediction = model.predict(input_data).tolist()[0]

    # Return the prediction as a JSON response
    return jsonify({'Sales': prediction})


# Run the Flask app in debug mode
if __name__ == '__main__':
    superkart_api.run(debug=True)

## Dependencies File

In [ ]:
%%writefile backend_files/requirements.txt
pandas==2.2.2
numpy==2.0.2
scikit-learn==1.6.1
seaborn==0.13.2
joblib==1.4.2
xgboost==2.1.4
joblib==1.4.2
Werkzeug==2.2.2
flask==2.2.2
gunicorn==20.1.0
requests==2.32.3
uvicorn[standard]
streamlit==1.43.2

## Dockerfile

In [ ]:
%%writefile backend_files/Dockerfile
FROM python:3.9-slim

# Set the working directory inside the container
WORKDIR /app

# Copy all files from the current directory to the container's working directory
COPY . /app

# Install dependencies from the requirements file without using cache to reduce image size
RUN pip install --no-cache-dir --upgrade -r requirements.txt

# Define the command to start the application using Gunicorn with 4 worker processes
# - `-w 4`: Uses 4 worker processes for handling requests
# - `-b 0.0.0.0:7860`: Binds the server to port 7860 on all network interfaces
# - `app:app`: Runs the Flask app (assuming `app.py` contains the Flask instance named `app`)
CMD ["gunicorn", "-w", "4", "-b", "0.0.0.0:7860", "app:superkart_api"]

## Setting up a Hugging Face Docker Space for the Backend

In [ ]:
# Import the login function from the huggingface_hub library
from huggingface_hub import login
openai_hf_key = userdata.get('MyHuggingFace')
login(token=openai_hf_key)

# Import the create_repo function from the huggingface_hub library
from huggingface_hub import create_repo

In [ ]:
try:
    create_repo("SuperKartBackend1",
        repo_type="space",
        space_sdk="docker",
        private=False
    )
except Exception as e:
    # Handle potential errors during repository creation
    if "RepositoryAlreadyExistsError" in str(e):
        print("Repository already exists. Skipping creation.")
    else:
        print(f"Error creating repository: {e}")

## Uploading Files to Hugging Face Space (Docker Space)

In [ ]:
# for hugging face space authentication to upload files

access_key = os.environ.get("HF_TOKEN")
repo_id = "NextGenModelCraft/SuperKartBackend1"

# Login to Hugging Face platform with the access token
login(token=access_key)

# Initialize the API
api = HfApi()

# Upload Streamlit app files stored in the folder called deployment_files
api.upload_folder(
    folder_path="backend_files",
    repo_id=repo_id,  # Hugging face space id
    repo_type="space",  # Hugging face repo type "space"
)

# **Deployment - Frontend**

## Points to note before executing the below cells
- Create a Streamlit space on Hugging Face by following the instructions provided on the content page titled **`Creating Spaces and Adding Secrets in Hugging Face`** from Week 1

## Streamlit for Interactive UI

In [ ]:
# Create a folder for storing the files needed for frontend UI deployment
os.makedirs("frontend_files", exist_ok=True)

In [ ]:
%%writefile frontend_files/app.py
import streamlit as st
import pandas as pd
import requests
import json

# --- Streamlit UI Setup ---
st.set_page_config(page_title="SuperKart Sales Predictor", layout="centered")
st.title("🛒 SuperKart Sales Predictor")
st.write("Enter product and store details to forecast sales revenue.")

# Backend API endpoint (placeholder for now)
# Replace with your deployed backend URL
BACKEND_URL = "https://NextGenModelCraft-SuperKartBackend.hf.space/v1/predict"

# --- Input Fields ---
st.header("Product Details")
product_weight = st.number_input("Product Weight (kg)", min_value=0.1, max_value=50.0, value=12.0, step=0.1)

product_sugar_content = st.selectbox(
    "Product Sugar Content",
    ("Low Sugar", "Regular", "No Sugar")
)

product_allocated_area = st.number_input("Product Allocated Area Ratio", min_value=0.001, max_value=0.5, value=0.05, format="%.3f", step=0.001)

product_mrp = st.number_input("Product MRP (Maximum Retail Price)", min_value=1.0, max_value=500.0, value=150.0, step=0.1)

product_id_char = st.selectbox(
    "Product ID Category (First two chars of Product_Id)",
    ("FD", "NC", "DR")
)

product_type_category = st.selectbox(
    "Product Type Category",
    ("Non Perishables", "Perishables")
)

st.header("Store Details")
store_size = st.selectbox(
    "Store Size",
    ("Medium", "High", "Small")
)

store_location_city_type = st.selectbox(
    "Store Location City Type",
    ("Tier 1", "Tier 2", "Tier 3")
)

store_type = st.selectbox(
    "Store Type",
    ("Supermarket Type1", "Supermarket Type2", "Departmental Store", "Food Mart")
)

store_age_years = st.number_input("Store Age (Years)", min_value=1, max_value=100, value=20, step=1)

# --- Prediction Button ---
if st.button("Forecast Sales"):
    # Prepare input data as a dictionary
    input_data = {
        "Product_Weight": product_weight,
        "Product_Sugar_Content": product_sugar_content,
        "Product_Allocated_Area": product_allocated_area,
        "Product_MRP": product_mrp,
        "Store_Size": store_size,
        "Store_Location_City_Type": store_location_city_type,
        "Store_Type": store_type,
        "Product_Id_char": product_id_char,
        "Store_Age_Years": store_age_years,
        "Product_Type_Category": product_type_category,
    }

    # Convert to JSON string
    json_data = json.dumps(input_data)

    st.write(f"Sending request to: {BACKEND_URL}")
    st.json(input_data)

    try:
      response = requests.post("https://NextGenModelCraft-SuperKartBackend.hf.space/v1/predict", json=input_data)
    # Check if response is successful
      if response.status_code == 200:
          # Check if response has content before parsing
          if response.text.strip():
              try:
                  prediction = response.json().get("prediction")
                  if prediction is not None:
                      st.success(f"Predicted Sales Revenue: ₹{prediction:,.2f}")
                  else:
                      st.error("Backend returned empty prediction")
              except requests.exceptions.JSONDecodeError:
                  st.error(f"Backend returned invalid JSON: {response.text[:200]}")
          else:
              st.error("Backend returned empty response")
      else:
          st.error(f"Error from backend: {response.status_code} - {response.text}")

    except requests.exceptions.ConnectionError:
        st.error("Could not connect to the backend API. Please ensure the backend is running and the URL is correct.")
    except requests.exceptions.Timeout:
        st.error("Request timed out. The backend is taking too long to respond.")
    except requests.exceptions.RequestException as e:
        st.error(f"Request failed: {e}")
    except Exception as e:
        st.error(f"An unexpected error occurred: {e}")

## Dependencies File

In [ ]:
%%writefile frontend_files/requirements.txt
requests==2.32.3
streamlit==1.45.0

## DockerFile

In [ ]:
%%writefile frontend_files/Dockerfile
# Use a minimal base image with Python 3.9 installed
FROM python:3.9-slim

# Set the working directory inside the container to /app
WORKDIR /app

# Copy all files from the current directory on the host to the container's /app directory
COPY . /app

# Install Python dependencies listed in requirements.txt
RUN pip3 install -r requirements.txt

# Define the command to run the Streamlit app on port 8501 and make it accessible externally
CMD ["streamlit", "run", "app.py", "--server.port=7860", "--server.address=0.0.0.0", "--server.enableXsrfProtection=false"]

# NOTE: Disable XSRF protection for easier external access in order to make batch predictions

## Uploading Files to Hugging Face Space (Streamlit Space)

In [ ]:
try:
    create_repo("SuperKartFrontend1",
        repo_type="space",  # Specify the repository type as "space"
        space_sdk="docker",  # Specify the space SDK as "docker"
        private=False  # Set to True if you want the space to be private
    )
except Exception as e:
    # Handle potential errors during repository creation
    if "RepositoryAlreadyExistsError" in str(e):
        print("Repository already exists. Skipping creation.")
    else:
        print(f"Error creating repository: {e}")

In [ ]:

access_key = os.environ.get("HF_TOKEN")
repo_id = "NextGenModelCraft/SuperKartFrontend1"

# Login to Hugging Face platform with the access token
login(token=access_key)

# Initialize the API
api = HfApi()

# Upload Streamlit app files stored in the folder called deployment_files
api.upload_folder(
    folder_path="frontend_files",
    repo_id=repo_id,  # Hugging face space id
    repo_type="space",  # Hugging face repo type "space"
)

# **Actionable Insights and Business Recommendations**

* We observe that Low sugar products have more sales followed by regular products which shows us the demand for categories of foods.
* Tier-city customization will help identify cities with faster growth and allocate more promotional budgets and premium assortments there.
* Store layout decisions helps allocate shelf space to high-sales categories to maximize revenue per square foot.
* Alert-based decision making where alerts are triggered for large sales/forecast deviations to prompt rapid action.
* Continuous model improvement need to be done by retraining models periodically with new data to adapt to changing consumer behavior.
* Tuning of the models does not guarantee better performance—both tuned models perform worse.
* XGB and Gradient (untuned) are the most reliable choices.